# Clustering (et exploitation des données géographiques)

Dans cette partie nous exploitons les données géographiques et réalisons le clustering. 

Nous exportons des statistiques par commune pour croisement avec les données de vote. 

In [ ]:
from BDTopo_fonctions import load_gpkg
gdf=load_gpkg("Sitadel/df_clustering_fulldep_1000m3.gpkg") #BASE ENREGISTREE AVEC SEULES ADRESSES PRECISEMENT LOCALISABLES (NUM VOIE = 1/3 SITADEL)

Téléchargement depuis mgarbe/Sitadel/df_clustering_fulldep_1000m3.gpkg ...
Chargement réussi (306523 lignes)


In [ ]:
#PB DANS GDF : ON A DES OBS EN DOUBLE (VIENT SUREMENT DE TEMP_STOCK2013 ??)

In [177]:
import importlib
import clustering_fonctions
importlib.reload(clustering_fonctions)
from clustering_fonctions import run_dbscan_parallele
temp=run_dbscan_parallele(gdf,700,10)

In [178]:
temp[(temp["cluster_id_2025"]==-1) & (temp["Base"]=="Sitadel")]

,Base,DEP_CODE,COMM,DATE_REELLE_AUTORISATION,DATE_REELLE_DOC,DATE_REELLE_DAACT,SURF_CREEE,I_EXTENSION,DESTINATION_PRINCIPALE,ID,...,cluster_id_2016,cluster_id_2017,cluster_id_2018,cluster_id_2019,cluster_id_2020,cluster_id_2021,cluster_id_2022,cluster_id_2023,cluster_id_2024,cluster_id_2025
7460,Sitadel,2,2560,25/04/2014,02/06/2014,None,1085.0,False,6,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
7469,Sitadel,2,2173,07/10/2015,07/03/2016,15/07/2016,1352.0,False,4,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
7556,Sitadel,2,2120,17/02/2016,18/02/2016,30/03/2018,1522.0,True,6,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
7559,Sitadel,2,2738,11/12/2017,15/02/2018,24/06/2019,1869.0,False,6,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
7573,Sitadel,2,2381,25/05/2018,20/08/2018,25/03/2019,3885.0,False,6,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305732,Sitadel,95,95428,06/02/2019,None,None,2790.0,False,4,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
306274,Sitadel,95,95658,31/03/2023,None,None,1784.0,True,8,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
306304,Sitadel,95,95652,07/02/2023,None,None,1827.0,False,8,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1
306341,Sitadel,95,95026,24/11/2023,21/11/2024,None,1130.0,False,8,None,...,-1,-1.0,-1.0,-1,-1,-1.0,-1,-1.0,-1,-1


In [179]:
subset =temp[(temp["cluster_id_2025"]==-1) & (temp["Base"]=="Sitadel")].copy()

# Reprojection vers WGS84 (latitude / longitude)
subset = subset.to_crs(epsg=4326)

# Calcul du centroïde
subset["centroid"] = subset.geometry.centroid
subset["lat"] = subset.centroid.y
subset["lon"] = subset.centroid.x

subset[['lat','lon','SURF_CREEE','Annee_REF']].sample(20)

/tmp/ipykernel_2190/2112594673.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["centroid"] = subset.geometry.centroid
/tmp/ipykernel_2190/2112594673.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["lat"] = subset.centroid.y
/tmp/ipykernel_2190/2112594673.py:9: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  subset["lon"] = subset.centroid.x


,lat,lon,SURF_CREEE,Annee_REF
275009,46.996711,-1.331199,18579.0,2019
282445,48.362596,6.302324,1214.0,2017
195179,50.865337,1.926051,1475.0,2017
128837,47.581315,2.763803,2287.0,2018
23548,43.230340,3.048415,2377.0,2017
114817,47.708837,1.604374,1170.0,2023
274867,46.672735,-1.456212,1285.0,2017
264523,43.828731,2.339976,1531.0,2025
264475,43.792365,1.745178,3577.0,2021
225211,47.797865,6.326240,1142.0,2020


In [ ]:
# Cluster_id présents dans Sitadel
sitadel_ids = temp.loc[temp["Base"]=="Sitadel", "cluster_id_2025"].unique()

# Comptage des occurrences de tous les bâtiments pour ces cluster_id
vc = temp["cluster_id_2025"].value_counts()

# Filtrer uniquement les clusters présents dans Sitadel
vc_sitadel = vc[vc.index.isin(sitadel_ids)]

# Nombre de clusters de taille 
vc_sitadel[vc_sitadel.index > 0]

cluster_id_2025
5900010    3732
9300001    2764
6900002    2367
9400001    2038
9200003    1123
           ... 
4900276       3
6200515       3
6200513       3
2500107       3
2500109       3
Name: count, Length: 1005, dtype: int64

In [168]:
# CLUSTER TOTAL, TOUS BAT EXISTANTS OU AUTORISES, ANNEE PAR ANNEE 
import pandas as pd
temp_sorted = temp.sort_values("Annee_REF")
results = []
for annee in sorted(temp_sorted["Annee_REF"].unique()):
    mask = temp_sorted["Annee_REF"] <= annee  # toutes les années <= annee
    cluster_col = temp_sorted.loc[mask, "cluster_id"]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    results.append({
        "Annee": annee,
        "pct_cluster_cumule": pct_cluster
    })
df_cumule = pd.DataFrame(results)
print(df_cumule)

    Annee  pct_cluster_cumule
0    2013           73.981335
1    2014           73.691219
2    2015           73.310854
3    2016           72.952130
4    2017           72.785524
5    2018           72.718488
6    2019           74.566734
7    2020           74.578316
8    2021           74.645909
9    2022           74.778250
10   2023           74.851406
11   2024           74.902310
12   2025           74.959138


In [174]:
#PC QUI EN 2025 SONT DANS UN CLUSTER
seuil=1000
temp[(temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>seuil)].groupby("Annee_REF")["cluster_id"].apply(lambda x: (x > 0).sum() / len(x) * 100)

Annee_REF
2014    94.141414
2015    95.155709
2016    95.054096
2017    94.222222
2018    94.685315
2019    95.077356
2020    94.354839
2021    93.685567
2022    93.804348
2023    93.136095
2024    92.335766
2025    90.729483
Name: cluster_id, dtype: float64

In [172]:
#PC QUI SONT DANS UN CLUSTER AU MOMENT DE AUTORISATION 

temp_sitadel = temp[(temp["Base"]=="Sitadel") & (temp["SURF_CREEE"]>seuil)]

for annee in sorted(temp_sitadel["Annee_REF"].unique()):
    col = f"cluster_id_{annee}"
    mask = temp_sitadel["Annee_REF"] == annee
    cluster_col = temp_sitadel.loc[mask, col]
    pct_cluster = (cluster_col > 0).sum() / len(cluster_col) * 100
    print(f"Année {annee}: {pct_cluster:.1f}% des bâtiments clusterisés")

Année 2014: 100.0% des bâtiments clusterisés
Année 2015: 94.0% des bâtiments clusterisés
Année 2016: 95.3% des bâtiments clusterisés
Année 2017: 98.7% des bâtiments clusterisés
Année 2018: 97.5% des bâtiments clusterisés
Année 2019: 94.9% des bâtiments clusterisés
Année 2020: 97.6% des bâtiments clusterisés
Année 2021: 92.7% des bâtiments clusterisés
Année 2022: 95.8% des bâtiments clusterisés
Année 2023: 94.6% des bâtiments clusterisés
Année 2024: 96.7% des bâtiments clusterisés
Année 2025: 98.0% des bâtiments clusterisés


In [173]:
annees = sorted(temp["Annee_REF"].unique())
result_list = []
for annee in annees:
    col = f"cluster_id_{annee}"
    if col in temp.columns:
        # compter les clusters uniques non-négatifs
        nb_clusters = temp[col][temp[col] > 0].nunique()
        result_list.append({"Annee_REF": annee, "nb_clusters": nb_clusters})

df_clusters_par_annee = pd.DataFrame(result_list)
print(df_clusters_par_annee)

    Annee_REF  nb_clusters
0        2013        13200
1        2014        13548
2        2015        13804
3        2016        14133
4        2017        14297
5        2018        14495
6        2019        16679
7        2020        16948
8        2021        16933
9        2022        17148
10       2023        17205
11       2024        17326
12       2025        17431
